# Sephora Core Facial Moisturizer SQL Analysis

## Business Objective

Identify where observed customer dissatisfaction is concentrated within a
historical Sephora facial-moisturizer review sample across brands, products,
price bands, and self-reported skin types. Assess whether the findings are
robust over time before using an LLM to explain the underlying complaint themes.

## Business Questions

1. How do positive, mixed, and negative review rates differ across brands, and where is dissatisfaction most concentrated?
2. Are the brand-level dissatisfaction signals driven by specific high-volume
   products?
3. How does observed satisfaction differ across price bands?
4. How do negative-review rates differ across self-reported skin types?
5. How have ratings and negative-review rates changed over time, and do the
   main patterns persist within a common recent period?

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sqlite3
import pandas as pd
from pathlib import Path

database_path = Path(
    "/content/drive/MyDrive/"
    "sephora-moisturizer-insights/"
    "sephora_reviews.db"
)

if not database_path.exists():
    raise FileNotFoundError(
        f"Database not found: {database_path}"
    )

conn = sqlite3.connect(database_path)

print("Connected to:", database_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Connected to: /content/drive/MyDrive/sephora-moisturizer-insights/sephora_reviews.db


In [ ]:
table_check = pd.read_sql_query("""
SELECT
    name AS table_name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", conn)

display(table_check)

,table_name
0,moisturizer_reviews
1,moisturizer_reviews_clean
2,moisturizer_reviews_final
3,products_raw
4,reviews_all
5,reviews_part_1
6,reviews_part_2
7,reviews_part_3
8,reviews_part_4
9,reviews_part_5


In [ ]:
baseline_check = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_reviews,
    COUNT(DISTINCT product_id) AS unique_skus,
    COUNT(DISTINCT product_family_id)
        AS unique_product_families,
    COUNT(DISTINCT catalog_brand_name)
        AS unique_brands,

    SUM(is_negative) AS negative_reviews,

    ROUND(
        100.0 * AVG(CAST(is_negative AS REAL)),
        2
    ) AS negative_rate_pct

FROM moisturizer_reviews_final;
""", conn)

display(baseline_check)

,total_reviews,unique_skus,unique_product_families,unique_brands,negative_reviews,negative_rate_pct
0,39366,70,68,46,2999,7.62


## Business Question 1: How do positive, mixed, and negative review rates differ across brands, and where is dissatisfaction most concentrated?

Brand-level satisfaction is evaluated using the full rating distribution:

- **Positive:** 4–5 stars
- **Mixed:** 3 stars
- **Negative:** 1–2 stars

The main comparison includes brands with at least 300 reviews. At the cohort
negative-review rate, this threshold corresponds to an approximate margin of
error of three percentage points and reduces instability from very small
samples.

The overall cohort rates—85.40% positive, 6.99% mixed, and 7.62% negative——are
used as benchmarks. A positive rate gap means that the brand's corresponding
rate is higher than the overall cohort rate; a negative gap means it is lower.

95% confidence intervals are included for negative-review rates to show
the precision of each estimate. Overlapping intervals indicate that the exact
ranking between brands may be uncertain.

Negative-review share measures how much each brand contributes to all negative
reviews in the cohort. `excess_negatives_vs_benchmark` estimates how many more
negative reviews were observed than expected if the brand had matched the
overall 7.62% negative-review rate. This is a benchmark comparison, not an
estimate of preventable complaints.

Recommendation rate is treated as a secondary metric because recommendation
response coverage varies across brands. These historical review results are
used as diagnostic signals and require product-level investigation before
brand-level recommendations are made.

In [ ]:
brand_performance = pd.read_sql_query("""
WITH overall_benchmark AS (
    SELECT
        100.0 * AVG(
            CASE
                WHEN rating >= 4 THEN 1.0
                ELSE 0.0
            END
        ) AS overall_positive_rate,

        100.0 * AVG(
            CASE
                WHEN rating = 3 THEN 1.0
                ELSE 0.0
            END
        ) AS overall_mixed_rate,

        100.0 * AVG(
            CAST(is_negative AS REAL)
        ) AS overall_negative_rate,

        SUM(is_negative)
            AS overall_negative_reviews

    FROM moisturizer_reviews_final
),

brand_metrics AS (
    SELECT
        catalog_brand_name AS brand_name,

        COUNT(*) AS total_reviews,

        COUNT(DISTINCT product_family_id)
            AS product_family_count,

        ROUND(AVG(rating), 2)
            AS avg_rating,

        /* Positive: 4–5 stars */
        SUM(
            CASE
                WHEN rating >= 4 THEN 1
                ELSE 0
            END
        ) AS positive_reviews,

        100.0 * AVG(
            CASE
                WHEN rating >= 4 THEN 1.0
                ELSE 0.0
            END
        ) AS positive_rate,

        /* Mixed: 3 stars */
        SUM(
            CASE
                WHEN rating = 3 THEN 1
                ELSE 0
            END
        ) AS mixed_reviews,

        100.0 * AVG(
            CASE
                WHEN rating = 3 THEN 1.0
                ELSE 0.0
            END
        ) AS mixed_rate,

        /* Negative: 1–2 stars */
        SUM(is_negative)
            AS negative_reviews,

        100.0 * AVG(
            CAST(is_negative AS REAL)
        ) AS negative_rate,

        /* Recommendation-response coverage */
        COUNT(is_recommended)
            AS recommendation_responses,

        100.0 * COUNT(is_recommended) / COUNT(*)
            AS recommendation_coverage,

        /* Recommendation rate among valid responses */
        100.0 * AVG(is_recommended)
            AS recommendation_rate

     FROM moisturizer_reviews_final

    WHERE catalog_brand_name IS NOT NULL
      AND TRIM(catalog_brand_name) <> ''

    GROUP BY catalog_brand_name
),

qualified_brands AS (
    SELECT *
    FROM brand_metrics
    WHERE total_reviews >= 300
)

SELECT
    brand_name,
    total_reviews,
    product_family_count,
    avg_rating,

    positive_reviews,

    ROUND(positive_rate, 2)
        AS positive_rate_pct,

    ROUND(
        positive_rate - overall_positive_rate,
        2
    ) AS positive_rate_gap_pp,

    mixed_reviews,

    ROUND(mixed_rate, 2)
        AS mixed_rate_pct,

    ROUND(
        mixed_rate - overall_mixed_rate,
        2
    ) AS mixed_rate_gap_pp,

    negative_reviews,

    ROUND(negative_rate, 2)
        AS negative_rate_pct,

    ROUND(
        negative_rate - overall_negative_rate,
        2
    ) AS negative_rate_gap_pp,

    ROUND(
        100.0 * negative_reviews
        / overall_negative_reviews,
        2
    ) AS negative_review_share_pct,

    ROUND(
        total_reviews
        * (negative_rate - overall_negative_rate)
        / 100.0,
        0
    ) AS excess_negatives_vs_benchmark,

    recommendation_responses,

    ROUND(recommendation_coverage, 2)
        AS recommendation_coverage_pct,

    ROUND(recommendation_rate, 2)
        AS recommendation_rate_pct,

    /* Validation: should equal approximately 100% */
    ROUND(
        positive_rate
        + mixed_rate
        + negative_rate,
        2
    ) AS rate_total_pct

FROM qualified_brands
CROSS JOIN overall_benchmark

ORDER BY
    negative_rate_pct DESC,
    total_reviews DESC;
""", conn)

display(brand_performance)

,brand_name,total_reviews,product_family_count,avg_rating,positive_reviews,positive_rate_pct,positive_rate_gap_pp,mixed_reviews,mixed_rate_pct,mixed_rate_gap_pp,negative_reviews,negative_rate_pct,negative_rate_gap_pp,negative_review_share_pct,excess_negatives_vs_benchmark,recommendation_responses,recommendation_coverage_pct,recommendation_rate_pct,rate_total_pct
0,Farmacy,1886,2,4.17,1437,76.19,-9.20,173,9.17,2.19,276,14.63,7.02,9.20,132.0,1786,94.70,76.76,100.0
1,Biossance,2840,2,4.17,2214,77.96,-7.44,273,9.61,2.63,353,12.43,4.81,11.77,137.0,2816,99.15,79.72,100.0
2,OLEHENRIKSEN,2960,1,4.34,2496,84.32,-1.07,191,6.45,-0.53,273,9.22,1.60,9.10,47.0,2960,100.00,86.62,100.0
3,Skinfix,771,1,4.52,677,87.81,2.41,30,3.89,-3.09,64,8.30,0.68,2.13,5.0,771,100.00,88.33,100.0
4,First Aid Beauty,9356,4,4.48,8083,86.39,1.00,509,5.44,-1.55,764,8.17,0.55,25.48,51.0,3515,37.57,83.78,100.0
5,Murad,641,2,4.46,557,86.90,1.50,36,5.62,-1.37,48,7.49,-0.13,1.60,-1.0,641,100.00,89.08,100.0
6,Summer Fridays,1400,2,4.48,1204,86.00,0.60,94,6.71,-0.27,102,7.29,-0.33,3.40,-5.0,1400,100.00,88.29,100.0
7,CLINIQUE,6004,9,4.35,5078,84.58,-0.82,530,8.83,1.84,396,6.60,-1.02,13.20,-61.0,2916,48.57,88.03,100.0
8,Tatcha,2198,2,4.55,1956,88.99,3.59,114,5.19,-1.80,128,5.82,-1.79,4.27,-39.0,2198,100.00,89.90,100.0
9,Dr. Jart+,645,1,4.50,569,88.22,2.82,41,6.36,-0.63,35,5.43,-2.19,1.17,-14.0,645,100.00,90.70,100.0


In [ ]:
import math
import pandas as pd

def calculate_negative_wilson_interval(row, z=1.96):
    total = int(row["total_reviews"])
    negative = int(row["negative_reviews"])
    rate = negative / total

    denominator = 1 + z**2 / total

    center = (
        rate + z**2 / (2 * total)
    ) / denominator

    half_width = (
        z
        * math.sqrt(
            rate * (1 - rate) / total
            + z**2 / (4 * total**2)
        )
        / denominator
    )

    return pd.Series({
        "negative_ci_lower_pct":
            round(100 * (center - half_width), 2),

        "negative_ci_upper_pct":
            round(100 * (center + half_width), 2)
    })

negative_confidence_intervals = brand_performance.apply(
    calculate_negative_wilson_interval,
    axis=1
)

brand_performance = pd.concat(
    [
        brand_performance,
        negative_confidence_intervals
    ],
    axis=1
)

display(brand_performance)

,brand_name,total_reviews,product_family_count,avg_rating,positive_reviews,positive_rate_pct,positive_rate_gap_pp,mixed_reviews,mixed_rate_pct,mixed_rate_gap_pp,...,negative_rate_pct,negative_rate_gap_pp,negative_review_share_pct,excess_negatives_vs_benchmark,recommendation_responses,recommendation_coverage_pct,recommendation_rate_pct,rate_total_pct,negative_ci_lower_pct,negative_ci_upper_pct
0,Farmacy,1886,2,4.17,1437,76.19,-9.20,173,9.17,2.19,...,14.63,7.02,9.20,132.0,1786,94.70,76.76,100.0,13.11,16.30
1,Biossance,2840,2,4.17,2214,77.96,-7.44,273,9.61,2.63,...,12.43,4.81,11.77,137.0,2816,99.15,79.72,100.0,11.27,13.69
2,OLEHENRIKSEN,2960,1,4.34,2496,84.32,-1.07,191,6.45,-0.53,...,9.22,1.60,9.10,47.0,2960,100.00,86.62,100.0,8.23,10.32
3,Skinfix,771,1,4.52,677,87.81,2.41,30,3.89,-3.09,...,8.30,0.68,2.13,5.0,771,100.00,88.33,100.0,6.55,10.46
4,First Aid Beauty,9356,4,4.48,8083,86.39,1.00,509,5.44,-1.55,...,8.17,0.55,25.48,51.0,3515,37.57,83.78,100.0,7.63,8.74
5,Murad,641,2,4.46,557,86.90,1.50,36,5.62,-1.37,...,7.49,-0.13,1.60,-1.0,641,100.00,89.08,100.0,5.69,9.79
6,Summer Fridays,1400,2,4.48,1204,86.00,0.60,94,6.71,-0.27,...,7.29,-0.33,3.40,-5.0,1400,100.00,88.29,100.0,6.04,8.77
7,CLINIQUE,6004,9,4.35,5078,84.58,-0.82,530,8.83,1.84,...,6.60,-1.02,13.20,-61.0,2916,48.57,88.03,100.0,6.00,7.25
8,Tatcha,2198,2,4.55,1956,88.99,3.59,114,5.19,-1.80,...,5.82,-1.79,4.27,-39.0,2198,100.00,89.90,100.0,4.92,6.88
9,Dr. Jart+,645,1,4.50,569,88.22,2.82,41,6.36,-0.63,...,5.43,-2.19,1.17,-14.0,645,100.00,90.70,100.0,3.93,7.45


In [ ]:
# Rating segments should add up to approximately 100%
assert brand_performance["rate_total_pct"].between(
    99.99,
    100.01
).all()

# Recommendation coverage cannot exceed 100%
assert brand_performance[
    "recommendation_coverage_pct"
].between(0, 100).all()

# Confidence-interval lower bound must be below upper bound
assert (
    brand_performance["negative_ci_lower_pct"]
    <= brand_performance["negative_ci_upper_pct"]
).all()

print("Brand analysis validation passed.")

Brand analysis validation passed.


### Finding 1: Brand Satisfaction Patterns Differ by Both Rate and Scale

The final cohort consisted of 85.40% positive reviews, 6.99% mixed reviews,
and 7.62% negative reviews.

Among brands with at least 300 historical reviews, Farmacy showed the strongest
dissatisfaction signal. Its positive-review rate was 76.19%, approximately
9.20 percentage points below the cohort benchmark. Its negative-review rate
was 14.63% (95% CI: 13.11%–16.30%), approximately 7.02 percentage points above
the benchmark.

Biossance displayed a similarly meaningful pattern across a larger sample of
2,840 reviews and two product families. Its positive-review rate was 77.96%,
while its negative-review rate was 12.43% (95% CI: 11.27%–13.69%). It generated
approximately 137 negative reviews above the cohort-based expectation, compared
with approximately 132 for Farmacy.

First Aid Beauty represented a different type of priority. Its positive-review
rate was slightly above the cohort benchmark at 86.39%, while its negative-review
rate was only approximately 0.55 percentage points above the benchmark.
However, its 764 negative reviews represented approximately 25.48% of all
negative reviews in the cohort, making it the largest opportunity by observed
complaint volume. Its recommendation rate should be interpreted cautiously
because recommendation-response coverage was only 37.57%.

After cross-SKU review-pool deduplication, Fenty Skin retained 177 review
records and therefore did not meet the 300-review threshold for the main brand
comparison. Any Fenty result should be treated as directional rather than
included in the primary ranking.

These results identify diagnostic signals within this historical Sephora review
sample. They do not establish that an entire brand performs poorly. Product-
family analysis is required to determine whether elevated dissatisfaction is
broadly distributed or concentrated in specific products.

## Business Question 2: Are Brand-Level Dissatisfaction Signals Driven by Specific Product Families?

This analysis decomposes the priority-brand signals identified in Question 1
into product families. Related full-size, mini-size, and legacy variants are
combined using `product_family_id` to avoid treating syndicated review pools
as independent products.

A brand is selected for product-family investigation when it has at least 300
reviews and meets either of the following diagnostic criteria:

- Its negative-review rate is at least three percentage points above the cohort
  benchmark; or
- It contributes at least 20% of all negative reviews in the cohort.

All product families within the selected brands are retained, including
families with small samples. Wilson 95% confidence intervals are used to
distinguish stronger evidence from directional signals.

The analysis evaluates whether each product family:

- Accounts for a large share of its brand's review volume;
- Accounts for a large share of its brand's negative reviews;
- Has a negative-review rate above its brand and cohort benchmarks; and
- Produces more negative reviews than expected at the cohort benchmark rate.

In [ ]:
product_driver_analysis = pd.read_sql_query("""
WITH overall_benchmark AS (
    SELECT
        100.0 * AVG(
            CAST(is_negative AS REAL)
        ) AS overall_negative_rate,

        SUM(is_negative)
            AS overall_negative_reviews

    FROM moisturizer_reviews_final
),

brand_metrics AS (
    SELECT
        catalog_brand_name AS brand_name,

        COUNT(*) AS brand_total_reviews,

        SUM(is_negative)
            AS brand_negative_reviews,

        100.0 * AVG(
            CAST(is_negative AS REAL)
        ) AS brand_negative_rate

    FROM moisturizer_reviews_final

    WHERE catalog_brand_name IS NOT NULL
      AND TRIM(catalog_brand_name) <> ''

    GROUP BY catalog_brand_name
),

priority_brands AS (
    SELECT
        brand.brand_name,

        CASE
            WHEN
                brand.brand_negative_rate
                - benchmark.overall_negative_rate >= 3.0

                AND

                100.0 * brand.brand_negative_reviews
                / benchmark.overall_negative_reviews >= 20.0

            THEN 'High rate and high volume'

            WHEN
                brand.brand_negative_rate
                - benchmark.overall_negative_rate >= 3.0

            THEN 'High negative rate'

            WHEN
                100.0 * brand.brand_negative_reviews
                / benchmark.overall_negative_reviews >= 20.0

            THEN 'High complaint volume'
        END AS priority_reason

    FROM brand_metrics AS brand

    CROSS JOIN overall_benchmark AS benchmark

    WHERE brand.brand_total_reviews >= 300

      AND (
          brand.brand_negative_rate
          - benchmark.overall_negative_rate >= 3.0

          OR

          100.0 * brand.brand_negative_reviews
          / benchmark.overall_negative_reviews >= 20.0
      )
),

product_family_metrics AS (
    SELECT
        reviews.catalog_brand_name
            AS brand_name,

        reviews.product_family_id,

        reviews.product_family_name,

        COUNT(DISTINCT reviews.product_id)
            AS sku_count,

        COUNT(*) AS total_reviews,

        ROUND(
            AVG(reviews.rating),
            2
        ) AS avg_rating,

        SUM(
            CASE
                WHEN reviews.rating >= 4 THEN 1
                ELSE 0
            END
        ) AS positive_reviews,

        100.0 * AVG(
            CASE
                WHEN reviews.rating >= 4 THEN 1.0
                ELSE 0.0
            END
        ) AS positive_rate,

        SUM(
            CASE
                WHEN reviews.rating = 3 THEN 1
                ELSE 0
            END
        ) AS mixed_reviews,

        100.0 * AVG(
            CASE
                WHEN reviews.rating = 3 THEN 1.0
                ELSE 0.0
            END
        ) AS mixed_rate,

        SUM(reviews.is_negative)
            AS negative_reviews,

        100.0 * AVG(
            CAST(reviews.is_negative AS REAL)
        ) AS negative_rate

    FROM moisturizer_reviews_final AS reviews

    INNER JOIN priority_brands AS priority
        ON reviews.catalog_brand_name
           = priority.brand_name

    GROUP BY
        reviews.catalog_brand_name,
        reviews.product_family_id,
        reviews.product_family_name
)

SELECT
    family.brand_name,
    priority.priority_reason,

    family.product_family_id,
    family.product_family_name,
    family.sku_count,
    family.total_reviews,

    ROUND(
        100.0 * family.total_reviews
        / brand.brand_total_reviews,
        2
    ) AS product_family_review_share_of_brand_pct,

    family.avg_rating,

    family.positive_reviews,

    ROUND(
        family.positive_rate,
        2
    ) AS positive_rate_pct,

    family.mixed_reviews,

    ROUND(
        family.mixed_rate,
        2
    ) AS mixed_rate_pct,

    family.negative_reviews,

    ROUND(
        family.negative_rate,
        2
    ) AS negative_rate_pct,

    ROUND(
        brand.brand_negative_rate,
        2
    ) AS brand_negative_rate_pct,

    ROUND(
        family.negative_rate
        - brand.brand_negative_rate,
        2
    ) AS negative_rate_gap_vs_brand_pp,

    ROUND(
        family.negative_rate
        - benchmark.overall_negative_rate,
        2
    ) AS negative_rate_gap_vs_cohort_pp,

    ROUND(
        100.0 * family.negative_reviews
        / brand.brand_negative_reviews,
        2
    ) AS negative_share_of_brand_pct,

    ROUND(
        family.total_reviews
        * (
            family.negative_rate
            - benchmark.overall_negative_rate
        )
        / 100.0,
        0
    ) AS excess_negatives_vs_cohort,

    RANK() OVER (
        PARTITION BY family.brand_name
        ORDER BY family.negative_reviews DESC
    ) AS negative_volume_rank_in_brand,

    ROUND(
        family.positive_rate
        + family.mixed_rate
        + family.negative_rate,
        2
    ) AS rate_total_pct

FROM product_family_metrics AS family

INNER JOIN brand_metrics AS brand
    ON family.brand_name = brand.brand_name

INNER JOIN priority_brands AS priority
    ON family.brand_name = priority.brand_name

CROSS JOIN overall_benchmark AS benchmark

ORDER BY
    family.brand_name,
    family.negative_reviews DESC,
    family.total_reviews DESC;
""", conn)

display(product_driver_analysis)

,brand_name,priority_reason,product_family_id,product_family_name,sku_count,total_reviews,product_family_review_share_of_brand_pct,avg_rating,positive_reviews,positive_rate_pct,...,mixed_rate_pct,negative_reviews,negative_rate_pct,brand_negative_rate_pct,negative_rate_gap_vs_brand_pp,negative_rate_gap_vs_cohort_pp,negative_share_of_brand_pct,excess_negatives_vs_cohort,negative_volume_rank_in_brand,rate_total_pct
0,Biossance,High negative rate,BIOSSANCE_OMEGA_REPAIR_FAMILY,Squalane + Omega Repair Deep Hydration Moistur...,2,2126,74.86,4.20,1688,79.40,...,9.31,240,11.29,12.43,-1.14,3.67,67.99,78.0,1,100.0
1,Biossance,High negative rate,P416561,Squalane + Probiotic Balancing Gel Moisturizer,1,714,25.14,4.09,526,73.67,...,10.50,113,15.83,12.43,3.40,8.21,32.01,59.0,2,100.0
2,Farmacy,High negative rate,P458209,Daily Greens Oil-Free Gel Moisturizer with Mor...,1,1356,71.90,4.06,989,72.94,...,10.47,225,16.59,14.63,1.96,8.97,81.52,122.0,1,100.0
3,Farmacy,High negative rate,P414293,Honey Savior All-in-One Skin Repair Salve,1,530,28.10,4.44,448,84.53,...,5.85,51,9.62,14.63,-5.01,2.00,18.48,11.0,2,100.0
4,First Aid Beauty,High complaint volume,P248407,Ultra Repair Cream Intense Hydration,1,7543,80.62,4.52,6613,87.67,...,4.89,561,7.44,8.17,-0.73,-0.18,73.43,-14.0,1,100.0
5,First Aid Beauty,High complaint volume,P375534,Ultra Repair Face Moisturizer,1,989,10.57,4.20,766,77.45,...,9.81,126,12.74,8.17,4.57,5.12,16.49,51.0,2,100.0
6,First Aid Beauty,High complaint volume,P468821,Ultra Repair Firming Collagen Cream with Pepti...,1,535,5.72,4.54,476,88.97,...,3.36,41,7.66,8.17,-0.50,0.05,5.37,0.0,3,100.0
7,First Aid Beauty,High complaint volume,P455894,Ultra Repair Oil-Control Moisturizer,1,289,3.09,4.24,228,78.89,...,8.65,36,12.46,8.17,4.29,4.84,4.71,14.0,4,100.0


In [ ]:
product_confidence_intervals = (
    product_driver_analysis.apply(
        calculate_negative_wilson_interval,
        axis=1
    )
)

product_driver_analysis = pd.concat(
    [
        product_driver_analysis,
        product_confidence_intervals
    ],
    axis=1
)

display(product_driver_analysis)

,brand_name,priority_reason,product_family_id,product_family_name,sku_count,total_reviews,product_family_review_share_of_brand_pct,avg_rating,positive_reviews,positive_rate_pct,...,negative_rate_pct,brand_negative_rate_pct,negative_rate_gap_vs_brand_pp,negative_rate_gap_vs_cohort_pp,negative_share_of_brand_pct,excess_negatives_vs_cohort,negative_volume_rank_in_brand,rate_total_pct,negative_ci_lower_pct,negative_ci_upper_pct
0,Biossance,High negative rate,BIOSSANCE_OMEGA_REPAIR_FAMILY,Squalane + Omega Repair Deep Hydration Moistur...,2,2126,74.86,4.20,1688,79.40,...,11.29,12.43,-1.14,3.67,67.99,78.0,1,100.0,10.01,12.70
1,Biossance,High negative rate,P416561,Squalane + Probiotic Balancing Gel Moisturizer,1,714,25.14,4.09,526,73.67,...,15.83,12.43,3.40,8.21,32.01,59.0,2,100.0,13.33,18.69
2,Farmacy,High negative rate,P458209,Daily Greens Oil-Free Gel Moisturizer with Mor...,1,1356,71.90,4.06,989,72.94,...,16.59,14.63,1.96,8.97,81.52,122.0,1,100.0,14.71,18.67
3,Farmacy,High negative rate,P414293,Honey Savior All-in-One Skin Repair Salve,1,530,28.10,4.44,448,84.53,...,9.62,14.63,-5.01,2.00,18.48,11.0,2,100.0,7.39,12.43
4,First Aid Beauty,High complaint volume,P248407,Ultra Repair Cream Intense Hydration,1,7543,80.62,4.52,6613,87.67,...,7.44,8.17,-0.73,-0.18,73.43,-14.0,1,100.0,6.87,8.05
5,First Aid Beauty,High complaint volume,P375534,Ultra Repair Face Moisturizer,1,989,10.57,4.20,766,77.45,...,12.74,8.17,4.57,5.12,16.49,51.0,2,100.0,10.81,14.96
6,First Aid Beauty,High complaint volume,P468821,Ultra Repair Firming Collagen Cream with Pepti...,1,535,5.72,4.54,476,88.97,...,7.66,8.17,-0.50,0.05,5.37,0.0,3,100.0,5.70,10.23
7,First Aid Beauty,High complaint volume,P455894,Ultra Repair Oil-Control Moisturizer,1,289,3.09,4.24,228,78.89,...,12.46,8.17,4.29,4.84,4.71,14.0,4,100.0,9.14,16.76


In [ ]:
# Every product-family rating distribution should total approximately 100%
assert product_driver_analysis[
    "rate_total_pct"
].between(99.99, 100.01).all()

# Fenty should be excluded because its final sample is below 300 reviews
assert "Fenty Skin" not in set(
    product_driver_analysis["brand_name"]
)

# Only the data-driven priority brands should remain
expected_priority_brands = {
    "Farmacy",
    "Biossance",
    "First Aid Beauty"
}

assert set(
    product_driver_analysis["brand_name"]
) == expected_priority_brands

# Product-family shares should sum to approximately 100% within each brand
brand_share_validation = (
    product_driver_analysis
    .groupby("brand_name")
    .agg(
        review_share_sum=(
            "product_family_review_share_of_brand_pct",
            "sum"
        ),
        negative_share_sum=(
            "negative_share_of_brand_pct",
            "sum"
        )
    )
    .round(2)
)

display(brand_share_validation)

assert brand_share_validation[
    "review_share_sum"
].between(99.98, 100.02).all()

assert brand_share_validation[
    "negative_share_sum"
].between(99.98, 100.02).all()

print("Product-family driver analysis validation passed.")

,review_share_sum,negative_share_sum
brand_name,,
Biossance,100.0,100.0
Farmacy,100.0,100.0
First Aid Beauty,100.0,100.0


Product-family driver analysis validation passed.


### Finding 2: Dissatisfaction Is Concentrated in Specific Product Families

Farmacy’s dissatisfaction signal was primarily driven by Daily Greens, which had a 16.59% negative-review rate, generated 81.52% of the brand’s negative reviews, and recorded approximately 122 negatives above the cohort expectation.

For Biossance, Probiotic Gel had the highest negative-review rate at 15.83%, while the larger Omega Repair family generated more excess negatives (78 versus 59). This represents a rate problem for Probiotic Gel and a scale problem for Omega Repair.

First Aid Beauty’s main Ultra Repair Cream was not a rate-driven concern: its 7.44% negative-review rate was slightly below the 7.62% cohort benchmark. Instead, Face Moisturizer showed the stronger signal at 12.74%. Oil-Control also reached 12.46%, but its smaller sample of 289 reviews makes the result directional.

Overall, the brand-level signals were concentrated in specific product families. Business action should target these families rather than entire brand portfolios.


## Business Question 3: How Does Observed Satisfaction Differ Across Price Bands?

Reviews are grouped using catalog price:

- Under $30

- $30 - $49.99

- $50 - $74.99

- $75 and above

Results are review-weighted, so popular products contribute more observations.
Product-family and SKU counts are shown to identify whether a price-band result
is driven by only a few products. Price differences are interpreted as
associations, not evidence that price causes satisfaction.

In [ ]:
price_band_audit = pd.read_sql_query("""
WITH priced_reviews AS (
    SELECT
        product_id,
        product_family_id,
        rating,
        is_negative,
        catalog_price_usd,

        CASE
            WHEN catalog_price_usd < 30
                THEN 'Under $30'
            WHEN catalog_price_usd < 50
                THEN '$30–$49.99'
            WHEN catalog_price_usd < 75
                THEN '$50–$74.99'
            ELSE '$75 and above'
        END AS price_band,

        CASE
            WHEN catalog_price_usd < 30 THEN 1
            WHEN catalog_price_usd < 50 THEN 2
            WHEN catalog_price_usd < 75 THEN 3
            ELSE 4
        END AS price_band_order

    FROM moisturizer_reviews_final

    WHERE catalog_price_usd IS NOT NULL
      AND catalog_price_usd > 0
)

SELECT
    price_band,
    COUNT(*) AS total_reviews,
    COUNT(DISTINCT product_id) AS sku_count,
    COUNT(DISTINCT product_family_id) AS product_family_count,

    ROUND(MIN(catalog_price_usd), 2) AS min_price,
    ROUND(MAX(catalog_price_usd), 2) AS max_price,

    ROUND(AVG(rating), 2) AS avg_rating,

    ROUND(
        100.0 * AVG(
            CASE WHEN rating >= 4 THEN 1.0 ELSE 0.0 END
        ),
        2
    ) AS positive_rate_pct,

    ROUND(
        100.0 * AVG(
            CASE WHEN rating = 3 THEN 1.0 ELSE 0.0 END
        ),
        2
    ) AS mixed_rate_pct,

    SUM(is_negative) AS negative_reviews,

    ROUND(
        100.0 * AVG(CAST(is_negative AS REAL)),
        2
    ) AS negative_rate_pct

FROM priced_reviews

GROUP BY
    price_band,
    price_band_order

ORDER BY price_band_order;
""", conn)

display(price_band_audit)

,price_band,total_reviews,sku_count,product_family_count,min_price,max_price,avg_rating,positive_rate_pct,mixed_rate_pct,negative_reviews,negative_rate_pct
0,Under $30,3453,9,9,14.0,28.0,4.38,83.67,7.70,298,8.63
1,$30–$49.99,23216,26,25,32.5,49.0,4.45,86.46,6.26,1691,7.28
2,$50–$74.99,10381,24,24,51.0,74.0,4.31,83.33,8.73,824,7.94
3,$75 and above,2316,11,11,75.0,200.0,4.45,86.57,5.40,186,8.03


In [ ]:
price_band_analysis = pd.read_sql_query("""
WITH overall_benchmark AS (
    SELECT
        100.0 * AVG(
            CASE WHEN rating >= 4 THEN 1.0 ELSE 0.0 END
        ) AS overall_positive_rate,

        100.0 * AVG(
            CASE WHEN rating = 3 THEN 1.0 ELSE 0.0 END
        ) AS overall_mixed_rate,

        100.0 * AVG(CAST(is_negative AS REAL))
            AS overall_negative_rate,

        SUM(is_negative) AS overall_negative_reviews

    FROM moisturizer_reviews_final
),

priced_reviews AS (
    SELECT
        *,
        CASE
            WHEN catalog_price_usd < 30 THEN 'Under $30'
            WHEN catalog_price_usd < 50 THEN '$30–$49.99'
            WHEN catalog_price_usd < 75 THEN '$50–$74.99'
            ELSE '$75 and above'
        END AS price_band,

        CASE
            WHEN catalog_price_usd < 30 THEN 1
            WHEN catalog_price_usd < 50 THEN 2
            WHEN catalog_price_usd < 75 THEN 3
            ELSE 4
        END AS price_band_order

    FROM moisturizer_reviews_final

    WHERE catalog_price_usd IS NOT NULL
      AND catalog_price_usd > 0
),

band_metrics AS (
    SELECT
        price_band,
        price_band_order,

        COUNT(*) AS total_reviews,
        COUNT(DISTINCT product_id) AS sku_count,
        COUNT(DISTINCT product_family_id)
            AS product_family_count,

        ROUND(MIN(catalog_price_usd), 2) AS min_price,
        ROUND(MAX(catalog_price_usd), 2) AS max_price,
        ROUND(AVG(rating), 2) AS avg_rating,

        SUM(
            CASE WHEN rating >= 4 THEN 1 ELSE 0 END
        ) AS positive_reviews,

        100.0 * AVG(
            CASE WHEN rating >= 4 THEN 1.0 ELSE 0.0 END
        ) AS positive_rate,

        SUM(
            CASE WHEN rating = 3 THEN 1 ELSE 0 END
        ) AS mixed_reviews,

        100.0 * AVG(
            CASE WHEN rating = 3 THEN 1.0 ELSE 0.0 END
        ) AS mixed_rate,

        SUM(is_negative) AS negative_reviews,

        100.0 * AVG(CAST(is_negative AS REAL))
            AS negative_rate,

        COUNT(is_recommended)
            AS recommendation_responses,

        100.0 * COUNT(is_recommended) / COUNT(*)
            AS recommendation_coverage,

        100.0 * AVG(is_recommended)
            AS recommendation_rate

    FROM priced_reviews

    GROUP BY
        price_band,
        price_band_order
)

SELECT
    band.price_band,
    band.total_reviews,
    band.sku_count,
    band.product_family_count,
    band.min_price,
    band.max_price,
    band.avg_rating,

    band.positive_reviews,
    ROUND(band.positive_rate, 2)
        AS positive_rate_pct,

    ROUND(
        band.positive_rate
        - benchmark.overall_positive_rate,
        2
    ) AS positive_rate_gap_pp,

    band.mixed_reviews,
    ROUND(band.mixed_rate, 2)
        AS mixed_rate_pct,

    band.negative_reviews,
    ROUND(band.negative_rate, 2)
        AS negative_rate_pct,

    ROUND(
        band.negative_rate
        - benchmark.overall_negative_rate,
        2
    ) AS negative_rate_gap_pp,

    ROUND(
        100.0 * band.negative_reviews
        / benchmark.overall_negative_reviews,
        2
    ) AS negative_review_share_pct,

    ROUND(
        band.total_reviews
        * (
            band.negative_rate
            - benchmark.overall_negative_rate
        ) / 100.0,
        0
    ) AS excess_negatives_vs_benchmark,

    band.recommendation_responses,

    ROUND(band.recommendation_coverage, 2)
        AS recommendation_coverage_pct,

    ROUND(band.recommendation_rate, 2)
        AS recommendation_rate_pct,

    ROUND(
        band.positive_rate
        + band.mixed_rate
        + band.negative_rate,
        2
    ) AS rate_total_pct

FROM band_metrics AS band
CROSS JOIN overall_benchmark AS benchmark

ORDER BY band.price_band_order;
""", conn)

display(price_band_analysis)

,price_band,total_reviews,sku_count,product_family_count,min_price,max_price,avg_rating,positive_reviews,positive_rate_pct,positive_rate_gap_pp,...,mixed_rate_pct,negative_reviews,negative_rate_pct,negative_rate_gap_pp,negative_review_share_pct,excess_negatives_vs_benchmark,recommendation_responses,recommendation_coverage_pct,recommendation_rate_pct,rate_total_pct
0,Under $30,3453,9,9,14.0,28.0,4.38,2889,83.67,-1.73,...,7.70,298,8.63,1.01,9.94,35.0,2971,86.04,86.84,100.0
1,$30–$49.99,23216,26,25,32.5,49.0,4.45,20072,86.46,1.06,...,6.26,1691,7.28,-0.33,56.39,-78.0,14735,63.47,87.09,100.0
2,$50–$74.99,10381,24,24,51.0,74.0,4.31,8651,83.33,-2.06,...,8.73,824,7.94,0.32,27.48,33.0,10276,98.99,86.40,100.0
3,$75 and above,2316,11,11,75.0,200.0,4.45,2005,86.57,1.18,...,5.40,186,8.03,0.41,6.20,10.0,2265,97.80,88.39,100.0


In [ ]:
price_band_confidence_intervals = (
    price_band_analysis.apply(
        calculate_negative_wilson_interval,
        axis=1
    )
)

price_band_analysis = pd.concat(
    [
        price_band_analysis,
        price_band_confidence_intervals
    ],
    axis=1
)

display(price_band_analysis)

,price_band,total_reviews,sku_count,product_family_count,min_price,max_price,avg_rating,positive_reviews,positive_rate_pct,positive_rate_gap_pp,...,negative_rate_pct,negative_rate_gap_pp,negative_review_share_pct,excess_negatives_vs_benchmark,recommendation_responses,recommendation_coverage_pct,recommendation_rate_pct,rate_total_pct,negative_ci_lower_pct,negative_ci_upper_pct
0,Under $30,3453,9,9,14.0,28.0,4.38,2889,83.67,-1.73,...,8.63,1.01,9.94,35.0,2971,86.04,86.84,100.0,7.74,9.61
1,$30–$49.99,23216,26,25,32.5,49.0,4.45,20072,86.46,1.06,...,7.28,-0.33,56.39,-78.0,14735,63.47,87.09,100.0,6.96,7.63
2,$50–$74.99,10381,24,24,51.0,74.0,4.31,8651,83.33,-2.06,...,7.94,0.32,27.48,33.0,10276,98.99,86.40,100.0,7.43,8.47
3,$75 and above,2316,11,11,75.0,200.0,4.45,2005,86.57,1.18,...,8.03,0.41,6.20,10.0,2265,97.80,88.39,100.0,6.99,9.21


In [ ]:
assert price_band_analysis["total_reviews"].sum() == 39366
assert price_band_analysis["negative_reviews"].sum() == 2999

assert price_band_analysis[
    "rate_total_pct"
].between(99.99, 100.01).all()

assert price_band_analysis[
    "recommendation_coverage_pct"
].between(0, 100).all()

print("Price-band analysis validation passed.")

Price-band analysis validation passed.


### Finding 3: Satisfaction Does Not Increase Consistently with Price

The relationship between price and satisfaction was not linear. The $30–$49.99
band had the lowest negative-review rate at 7.28%, while products under $30 had
the highest rate at 8.63%.

The $50–$74.99 band had the lowest positive-review rate and highest mixed-review
rate, indicating more moderate rather than strongly negative feedback. Products
priced at $75 or above had a high positive-review rate, but their 8.03% negative
rate and wider confidence interval did not show a clear premium-price advantage.

Overall, price alone did not reliably predict satisfaction. Differences may also
reflect the brands and product families represented within each price band.

## Business Question 4: How Do Review Outcomes Differ Across Self-Reported Skin Types?

Skin-type comparisons include only reviews with a reported skin type. Missing
values are excluded rather than classified as “Unknown” or imputed.

Positive, mixed, and negative rates are calculated within each skin-type group.
Wilson 95% confidence intervals are included to show estimation precision.
Because skin type is self-reported and product usage may differ across groups,
results are interpreted as associations rather than causal effects.

In [ ]:
skin_type_analysis = pd.read_sql_query("""
WITH reported_reviews AS (
    SELECT
        *,
        LOWER(TRIM(skin_type)) AS reported_skin_type

    FROM moisturizer_reviews_final

    WHERE skin_type IS NOT NULL
      AND TRIM(skin_type) <> ''
),

reported_benchmark AS (
    SELECT
        COUNT(*) AS reported_reviews,

        100.0 * AVG(
            CASE WHEN rating >= 4 THEN 1.0 ELSE 0.0 END
        ) AS overall_positive_rate,

        100.0 * AVG(
            CASE WHEN rating = 3 THEN 1.0 ELSE 0.0 END
        ) AS overall_mixed_rate,

        100.0 * AVG(CAST(is_negative AS REAL))
            AS overall_negative_rate,

        SUM(is_negative) AS overall_negative_reviews

    FROM reported_reviews
),

skin_metrics AS (
    SELECT
        reported_skin_type AS skin_type,

        COUNT(*) AS total_reviews,

        COUNT(DISTINCT product_family_id)
            AS product_family_count,

        ROUND(AVG(rating), 2) AS avg_rating,

        SUM(
            CASE WHEN rating >= 4 THEN 1 ELSE 0 END
        ) AS positive_reviews,

        100.0 * AVG(
            CASE WHEN rating >= 4 THEN 1.0 ELSE 0.0 END
        ) AS positive_rate,

        SUM(
            CASE WHEN rating = 3 THEN 1 ELSE 0 END
        ) AS mixed_reviews,

        100.0 * AVG(
            CASE WHEN rating = 3 THEN 1.0 ELSE 0.0 END
        ) AS mixed_rate,

        SUM(is_negative) AS negative_reviews,

        100.0 * AVG(CAST(is_negative AS REAL))
            AS negative_rate,

        COUNT(is_recommended)
            AS recommendation_responses,

        100.0 * COUNT(is_recommended) / COUNT(*)
            AS recommendation_coverage,

        100.0 * AVG(is_recommended)
            AS recommendation_rate

    FROM reported_reviews

    GROUP BY reported_skin_type
)

SELECT
    skin.skin_type,
    skin.total_reviews,
    skin.product_family_count,

    ROUND(
        100.0 * skin.total_reviews
        / benchmark.reported_reviews,
        2
    ) AS review_share_of_reported_pct,

    skin.avg_rating,

    skin.positive_reviews,
    ROUND(skin.positive_rate, 2)
        AS positive_rate_pct,

    ROUND(
        skin.positive_rate
        - benchmark.overall_positive_rate,
        2
    ) AS positive_rate_gap_pp,

    skin.mixed_reviews,
    ROUND(skin.mixed_rate, 2)
        AS mixed_rate_pct,

    skin.negative_reviews,
    ROUND(skin.negative_rate, 2)
        AS negative_rate_pct,

    ROUND(
        skin.negative_rate
        - benchmark.overall_negative_rate,
        2
    ) AS negative_rate_gap_pp,

    ROUND(
        100.0 * skin.negative_reviews
        / benchmark.overall_negative_reviews,
        2
    ) AS negative_review_share_pct,

    ROUND(
        skin.total_reviews
        * (
            skin.negative_rate
            - benchmark.overall_negative_rate
        ) / 100.0,
        0
    ) AS excess_negatives_vs_reported_benchmark,

    skin.recommendation_responses,

    ROUND(skin.recommendation_coverage, 2)
        AS recommendation_coverage_pct,

    ROUND(skin.recommendation_rate, 2)
        AS recommendation_rate_pct,

    ROUND(
        skin.positive_rate
        + skin.mixed_rate
        + skin.negative_rate,
        2
    ) AS rate_total_pct

FROM skin_metrics AS skin
CROSS JOIN reported_benchmark AS benchmark

ORDER BY negative_rate_pct DESC;
""", conn)

display(skin_type_analysis)

,skin_type,total_reviews,product_family_count,review_share_of_reported_pct,avg_rating,positive_reviews,positive_rate_pct,positive_rate_gap_pp,mixed_reviews,mixed_rate_pct,negative_reviews,negative_rate_pct,negative_rate_gap_pp,negative_review_share_pct,excess_negatives_vs_reported_benchmark,recommendation_responses,recommendation_coverage_pct,recommendation_rate_pct,rate_total_pct
0,oily,4508,60,12.68,4.37,3793,84.14,-1.25,346,7.68,369,8.19,0.63,13.74,29.0,3691,81.88,85.80,100.0
1,normal,4306,65,12.11,4.39,3680,85.46,0.07,289,6.71,337,7.83,0.27,12.55,12.0,3794,88.11,86.93,100.0
2,dry,7288,65,20.50,4.41,6216,85.29,-0.10,511,7.01,561,7.70,0.14,20.89,11.0,5959,81.76,86.88,100.0
3,combination,19447,66,54.70,4.41,16666,85.70,0.31,1363,7.01,1418,7.29,-0.26,52.81,-51.0,16292,83.78,87.24,100.0


In [ ]:
skin_type_confidence_intervals = (
    skin_type_analysis.apply(
        calculate_negative_wilson_interval,
        axis=1
    )
)

skin_type_analysis = pd.concat(
    [
        skin_type_analysis,
        skin_type_confidence_intervals
    ],
    axis=1
)

display(skin_type_analysis)

,skin_type,total_reviews,product_family_count,review_share_of_reported_pct,avg_rating,positive_reviews,positive_rate_pct,positive_rate_gap_pp,mixed_reviews,mixed_rate_pct,...,negative_rate_pct,negative_rate_gap_pp,negative_review_share_pct,excess_negatives_vs_reported_benchmark,recommendation_responses,recommendation_coverage_pct,recommendation_rate_pct,rate_total_pct,negative_ci_lower_pct,negative_ci_upper_pct
0,oily,4508,60,12.68,4.37,3793,84.14,-1.25,346,7.68,...,8.19,0.63,13.74,29.0,3691,81.88,85.80,100.0,7.42,9.02
1,normal,4306,65,12.11,4.39,3680,85.46,0.07,289,6.71,...,7.83,0.27,12.55,12.0,3794,88.11,86.93,100.0,7.06,8.67
2,dry,7288,65,20.50,4.41,6216,85.29,-0.10,511,7.01,...,7.70,0.14,20.89,11.0,5959,81.76,86.88,100.0,7.11,8.33
3,combination,19447,66,54.70,4.41,16666,85.70,0.31,1363,7.01,...,7.29,-0.26,52.81,-51.0,16292,83.78,87.24,100.0,6.93,7.67


In [ ]:
assert skin_type_analysis["total_reviews"].sum() == 35549

assert set(skin_type_analysis["skin_type"]) == {
    "combination",
    "dry",
    "normal",
    "oily"
}

assert skin_type_analysis[
    "rate_total_pct"
].between(99.99, 100.01).all()

assert skin_type_analysis[
    "recommendation_coverage_pct"
].between(0, 100).all()

print("Skin-type analysis validation passed.")

Skin-type analysis validation passed.


### Finding 4: Skin-Type Differences Are Modest

Oily-skin reviewers had the highest negative-review rate at 8.19%, while
combination-skin reviewers had the lowest rate at 7.29%. Oily skin also had the
lowest positive-review rate at 84.14%.

However, the differences were relatively small, and the 95% confidence
intervals overlapped. Dry- and normal-skin reviewers remained close to the
reported-skin benchmark.

Overall, oily-skin customers showed a modestly higher dissatisfaction signal,
but the evidence does not support broad product recommendations based on skin
type alone. Review-text analysis is needed to identify whether oily-skin
customers report distinct complaint themes.

## Business Question 5: How Have Ratings and Negative-Review Rates Changed Over Time?

Annual review outcomes are examined using average rating and positive, mixed,
and negative-review rates. Review volume, brand coverage, and product-family
coverage are shown because changes over time may reflect changes in the products
and brands represented in the dataset.

The final year, 2023, contains data only through March and is therefore treated
as a partial year. A recent common-period sensitivity analysis will be conducted
after identifying years with sufficient overlapping coverage for Farmacy,
Biossance, and First Aid Beauty.

In [ ]:
yearly_coverage_audit = pd.read_sql_query("""
WITH dated_reviews AS (
    SELECT
        CAST(
            SUBSTR(submission_time, 1, 4)
            AS INTEGER
        ) AS review_year,

        catalog_brand_name,
        product_family_id,
        rating,
        is_negative,
        submission_time

    FROM moisturizer_reviews_final

    WHERE submission_time IS NOT NULL
      AND TRIM(submission_time) <> ''
)

SELECT
    review_year,
    COUNT(*) AS total_reviews,
    COUNT(DISTINCT catalog_brand_name)
        AS brand_count,
    COUNT(DISTINCT product_family_id)
        AS product_family_count,

    MIN(submission_time) AS first_review_date,
    MAX(submission_time) AS last_review_date,

    ROUND(AVG(rating), 2) AS avg_rating,

    ROUND(
        100.0 * AVG(CAST(is_negative AS REAL)),
        2
    ) AS negative_rate_pct

FROM dated_reviews

GROUP BY review_year
ORDER BY review_year;
""", conn)

display(yearly_coverage_audit)

,review_year,total_reviews,brand_count,product_family_count,first_review_date,last_review_date,avg_rating,negative_rate_pct
0,2008,108,1,1,2008-08-30,2008-12-28,4.60,3.70
1,2009,382,2,2,2009-01-01,2009-12-31,4.56,4.45
2,2010,1134,2,2,2010-01-01,2010-12-31,4.62,3.53
3,2011,1024,2,2,2011-01-01,2011-12-31,4.63,3.81
4,2012,848,2,4,2012-01-01,2012-12-31,4.61,3.54
5,2013,1058,3,5,2013-01-01,2013-12-31,4.54,6.24
6,2014,1294,3,6,2014-01-01,2014-12-31,4.48,8.27
7,2015,1223,3,6,2015-01-01,2015-12-31,4.42,9.73
8,2016,1329,5,11,2016-01-01,2016-12-31,4.42,8.88
9,2017,1243,10,17,2017-01-01,2017-12-31,4.41,9.25


In [ ]:
common_period_audit = pd.read_sql_query("""
WITH dated_reviews AS (
    SELECT
        CAST(
            SUBSTR(submission_time, 1, 4)
            AS INTEGER
        ) AS review_year,

        catalog_brand_name AS brand_name,
        product_family_id

    FROM moisturizer_reviews_final

    WHERE submission_time IS NOT NULL
      AND TRIM(submission_time) <> ''

      AND catalog_brand_name IN (
          'Farmacy',
          'Biossance',
          'First Aid Beauty'
      )
),

brand_year_metrics AS (
    SELECT
        review_year,
        brand_name,
        COUNT(*) AS brand_reviews,
        COUNT(DISTINCT product_family_id)
            AS product_family_count

    FROM dated_reviews

    GROUP BY
        review_year,
        brand_name
)

SELECT
    review_year,

    COUNT(DISTINCT brand_name)
        AS brands_present,

    SUM(brand_reviews)
        AS combined_reviews,

    MIN(brand_reviews)
        AS smallest_brand_sample,

    MAX(brand_reviews)
        AS largest_brand_sample

FROM brand_year_metrics

GROUP BY review_year

HAVING COUNT(DISTINCT brand_name) = 3

ORDER BY review_year;
""", conn)

display(common_period_audit)

,review_year,brands_present,combined_reviews,smallest_brand_sample,largest_brand_sample
0,2017,3,907,49,720
1,2018,3,1247,151,655
2,2019,3,930,103,512
3,2020,3,2165,478,1004
4,2021,3,1541,283,876
5,2022,3,1654,180,1101
6,2023,3,219,25,120


In [ ]:
annual_trend_analysis = pd.read_sql_query("""
SELECT
    CAST(SUBSTR(submission_time, 1, 4) AS INTEGER)
        AS review_year,

    COUNT(*) AS total_reviews,
    COUNT(DISTINCT catalog_brand_name) AS brand_count,
    COUNT(DISTINCT product_family_id) AS product_family_count,

    ROUND(AVG(rating), 2) AS avg_rating,

    SUM(CASE WHEN rating >= 4 THEN 1 ELSE 0 END)
        AS positive_reviews,

    ROUND(
        100.0 * AVG(
            CASE WHEN rating >= 4 THEN 1.0 ELSE 0.0 END
        ),
        2
    ) AS positive_rate_pct,

    SUM(CASE WHEN rating = 3 THEN 1 ELSE 0 END)
        AS mixed_reviews,

    ROUND(
        100.0 * AVG(
            CASE WHEN rating = 3 THEN 1.0 ELSE 0.0 END
        ),
        2
    ) AS mixed_rate_pct,

    SUM(is_negative) AS negative_reviews,

    ROUND(
        100.0 * AVG(CAST(is_negative AS REAL)),
        2
    ) AS negative_rate_pct

FROM moisturizer_reviews_final

WHERE CAST(SUBSTR(submission_time, 1, 4) AS INTEGER)
      BETWEEN 2018 AND 2022

GROUP BY review_year
ORDER BY review_year;
""", conn)

display(annual_trend_analysis)

,review_year,total_reviews,brand_count,product_family_count,avg_rating,positive_reviews,positive_rate_pct,mixed_reviews,mixed_rate_pct,negative_reviews,negative_rate_pct
0,2018,5415,12,21,4.54,4896,90.42,265,4.89,254,4.69
1,2019,2708,15,21,4.34,2235,82.53,187,6.91,286,10.56
2,2020,6369,22,32,4.25,5174,81.24,604,9.48,591,9.28
3,2021,6023,33,45,4.45,5183,86.05,379,6.29,461,7.65
4,2022,7388,43,60,4.29,6098,82.54,687,9.30,603,8.16


In [ ]:
annual_confidence_intervals = (
    annual_trend_analysis.apply(
        calculate_negative_wilson_interval,
        axis=1
    )
)

annual_trend_analysis = pd.concat(
    [
        annual_trend_analysis,
        annual_confidence_intervals
    ],
    axis=1
)

display(annual_trend_analysis)

,review_year,total_reviews,brand_count,product_family_count,avg_rating,positive_reviews,positive_rate_pct,mixed_reviews,mixed_rate_pct,negative_reviews,negative_rate_pct,negative_ci_lower_pct,negative_ci_upper_pct
0,2018,5415,12,21,4.54,4896,90.42,265,4.89,254,4.69,4.16,5.29
1,2019,2708,15,21,4.34,2235,82.53,187,6.91,286,10.56,9.46,11.78
2,2020,6369,22,32,4.25,5174,81.24,604,9.48,591,9.28,8.59,10.02
3,2021,6023,33,45,4.45,5183,86.05,379,6.29,461,7.65,7.01,8.35
4,2022,7388,43,60,4.29,6098,82.54,687,9.30,603,8.16,7.56,8.81


In [ ]:
recent_brand_analysis = pd.read_sql_query("""
WITH recent_reviews AS (
    SELECT *
    FROM moisturizer_reviews_final

    WHERE CAST(SUBSTR(submission_time, 1, 4) AS INTEGER)
          BETWEEN 2020 AND 2022
),

recent_benchmark AS (
    SELECT
        100.0 * AVG(
            CASE WHEN rating >= 4 THEN 1.0 ELSE 0.0 END
        ) AS benchmark_positive_rate,

        100.0 * AVG(
            CASE WHEN rating = 3 THEN 1.0 ELSE 0.0 END
        ) AS benchmark_mixed_rate,

        100.0 * AVG(CAST(is_negative AS REAL))
            AS benchmark_negative_rate

    FROM recent_reviews
),

brand_metrics AS (
    SELECT
        catalog_brand_name AS brand_name,
        COUNT(*) AS total_reviews,
        COUNT(DISTINCT product_family_id)
            AS product_family_count,

        ROUND(AVG(rating), 2) AS avg_rating,

        SUM(CASE WHEN rating >= 4 THEN 1 ELSE 0 END)
            AS positive_reviews,

        100.0 * AVG(
            CASE WHEN rating >= 4 THEN 1.0 ELSE 0.0 END
        ) AS positive_rate,

        SUM(CASE WHEN rating = 3 THEN 1 ELSE 0 END)
            AS mixed_reviews,

        100.0 * AVG(
            CASE WHEN rating = 3 THEN 1.0 ELSE 0.0 END
        ) AS mixed_rate,

        SUM(is_negative) AS negative_reviews,

        100.0 * AVG(CAST(is_negative AS REAL))
            AS negative_rate

    FROM recent_reviews

    WHERE catalog_brand_name IN (
        'Farmacy',
        'Biossance',
        'First Aid Beauty'
    )

    GROUP BY catalog_brand_name
)

SELECT
    brand.brand_name,
    brand.total_reviews,
    brand.product_family_count,
    brand.avg_rating,

    brand.positive_reviews,
    ROUND(brand.positive_rate, 2)
        AS positive_rate_pct,

    ROUND(
        brand.positive_rate
        - benchmark.benchmark_positive_rate,
        2
    ) AS positive_rate_gap_pp,

    brand.mixed_reviews,
    ROUND(brand.mixed_rate, 2)
        AS mixed_rate_pct,

    brand.negative_reviews,
    ROUND(brand.negative_rate, 2)
        AS negative_rate_pct,

    ROUND(
        brand.negative_rate
        - benchmark.benchmark_negative_rate,
        2
    ) AS negative_rate_gap_pp

FROM brand_metrics AS brand
CROSS JOIN recent_benchmark AS benchmark

ORDER BY negative_rate_pct DESC;
""", conn)

display(recent_brand_analysis)

,brand_name,total_reviews,product_family_count,avg_rating,positive_reviews,positive_rate_pct,positive_rate_gap_pp,mixed_reviews,mixed_rate_pct,negative_reviews,negative_rate_pct,negative_rate_gap_pp
0,Farmacy,1467,2,4.09,1082,73.76,-9.43,154,10.50,231,15.75,7.38
1,Biossance,1961,2,4.11,1501,76.54,-6.65,198,10.10,262,13.36,4.99
2,First Aid Beauty,1932,4,4.26,1541,79.76,-3.43,135,6.99,256,13.25,4.88


In [ ]:
recent_brand_confidence_intervals = (
    recent_brand_analysis.apply(
        calculate_negative_wilson_interval,
        axis=1
    )
)

recent_brand_analysis = pd.concat(
    [
        recent_brand_analysis,
        recent_brand_confidence_intervals
    ],
    axis=1
)

display(recent_brand_analysis)

,brand_name,total_reviews,product_family_count,avg_rating,positive_reviews,positive_rate_pct,positive_rate_gap_pp,mixed_reviews,mixed_rate_pct,negative_reviews,negative_rate_pct,negative_rate_gap_pp,negative_ci_lower_pct,negative_ci_upper_pct
0,Farmacy,1467,2,4.09,1082,73.76,-9.43,154,10.50,231,15.75,7.38,13.97,17.70
1,Biossance,1961,2,4.11,1501,76.54,-6.65,198,10.10,262,13.36,4.99,11.93,14.94
2,First Aid Beauty,1932,4,4.26,1541,79.76,-3.43,135,6.99,256,13.25,4.88,11.81,14.84


In [ ]:
assert set(annual_trend_analysis["review_year"]) == {
    2018, 2019, 2020, 2021, 2022
}

assert set(recent_brand_analysis["brand_name"]) == {
    "Farmacy",
    "Biossance",
    "First Aid Beauty"
}

assert recent_brand_analysis["total_reviews"].ge(300).all()

print("Time-trend analysis validation passed.")

Time-trend analysis validation passed.


### Finding 5: Satisfaction Varied Over Time, but Priority-Brand Signals Persisted

Annual satisfaction did not follow a consistent upward or downward trend.
Negative-review rates increased from 4.69% in 2018 to 10.56% in 2019, then
declined to 7.65% in 2021 before rising slightly to 8.16% in 2022.

These annual comparisons should be interpreted cautiously because brand
coverage increased from 12 brands in 2018 to 43 in 2022, meaning that changes
may partly reflect shifts in product and brand composition.

Within the common 2020–2022 period, Farmacy had the highest negative-review
rate at 15.75%, followed by Biossance at 13.36% and First Aid Beauty at 13.25%.
All three exceeded the recent-period benchmark of approximately 8.37%.

Farmacy and Biossance therefore retained their full-period dissatisfaction
signals. First Aid Beauty’s stronger recent-period rate suggests that its
historical complaint-volume signal was also concentrated in more recent years.

In [ ]:
# Final notebook completion checks
assert int(baseline_check.loc[0, "total_reviews"]) == 39366
assert int(baseline_check.loc[0, "negative_reviews"]) == 2999

assert "Fenty Skin" not in set(
    product_driver_analysis["brand_name"]
)

assert price_band_analysis["total_reviews"].sum() == 39366
assert skin_type_analysis["total_reviews"].sum() == 35549

assert set(annual_trend_analysis["review_year"]) == {
    2018, 2019, 2020, 2021, 2022
}

assert set(recent_brand_analysis["brand_name"]) == {
    "Farmacy",
    "Biossance",
    "First Aid Beauty"
}

conn.close()

print("02_sql_analysis.ipynb completed successfully.")

02_sql_analysis.ipynb completed successfully.
